# Gradual typing adoption with mypy

> L3 investigation: stepping a small untyped project toward `--strict` without drowning in errors on day one.

Three mechanisms drive the adoption path on a real-ish module:

- `reveal_type()` to see what mypy infers at each point (and confirm it matches the mental model).
- `--check-untyped-defs` to catch body errors inside functions that still lack annotations.
- `# type: ignore[<code>]` scoping so suppressions stay narrow instead of blanket-silencing a whole file.

The baseline: a small scoreboard module that starts completely untyped, then gets ratcheted up to strict-clean one flag at a time.

## Step 1 — the silent pass gotcha

Start with a module that has no annotations at all. mypy's default behavior is to skip unannotated functions entirely, so obvious body errors slide through.

In [ ]:
# scoreboard.py — completely untyped, version 1

def parse_line(line):
    name, _, raw = line.partition(":")
    return name.strip(), int(raw)


def tally(lines):
    scores = {}
    for line in lines:
        name, points = parse_line(line)
        scores.setdefault(name, []).append(points)
    return scores


def best_player(scores):
    best = None
    best_total = -1
    for name, points in scores.items():
        total = sum(points)
        if total > best_total:
            best_total = total
            best = name
    return best


main = ["alice:3", "bob:7", "alice:2", "bob:1"]
print(best_player(tally(main)))

In [ ]:
# Run:  mypy scoreboard.py
# Result: Success: no issues found in 1 source file
#
# That's the silent-pass gotcha — unannotated functions are deliberately
# never type-checked, so body errors (like mixing str and int) slide through.
# The code is wrong-safe only by luck, not by verification.

## Step 2 — reveal_type to confirm inference

Before adding annotations, drop `reveal_type()` calls to see what mypy actually infers. This catches mismatches between the mental model and reality early.

In [ ]:
# scoreboard.py — version 2, with reveal_type probes
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from typing import reveal_type


def parse_line(line):
    name, _, raw = line.partition(":")
    if TYPE_CHECKING:
        reveal_type(name)  # what does mypy think `name` is?
    return name.strip(), int(raw)


def tally(lines):
    scores = {}
    for line in lines:
        name, points = parse_line(line)
        scores.setdefault(name, []).append(points)
    if TYPE_CHECKING:
        reveal_type(scores)  # inferred dict type?
    return scores


def best_player(scores):
    best = None
    best_total = -1
    for name, points in scores.items():
        total = sum(points)
        if total > best_total:
            best_total = total
            best = name
    return best


main = ["alice:3", "bob:7", "alice:2", "bob:1"]
print(best_player(tally(main)))

In [ ]:
# Run:  mypy scoreboard.py
#
# scoreboard.py:11: note: Revealed type is "str"
# scoreboard.py:22: note: Revealed type is "builtins.dict[builtins.str, builtins.list[builtins.int]]"
#
# Two things stand out:
# - `name` is correctly inferred as str from partition().
# - `scores` is dict[str, list[int]] — but only because the loop body
#   populated it. An empty `{}` literal would have been dict[str, Any]
#   without the annotation. The reveal_type confirms the narrowing worked.
#
# Note: reveal_type() lives behind `if TYPE_CHECKING:` because it only
# exists in mypy's world — Python raises NameError if it runs.
# The probes get removed once the real annotations are in place.

## Step 3 — `--check-untyped-defs` catches body errors

With inference confirmed, enable `--check-untyped-defs` (or `check_untyped_defs = true` in config). This makes mypy type-check the *bodies* of unannotated functions against the types it can infer, without yet requiring annotations on the signature.

In [ ]:
# Introduce a body error to see --check-untyped-defs catch it:
def best_player(scores):
    best = None
    best_total = -1
    for name, points in scores.items():
        total = sum(points)
        if total > best_total:
            best_total = total
            best = name
    return best + "!"  # ERROR: best can be None

In [ ]:
# Run:  mypy --check-untyped-defs scoreboard.py
#
# scoreboard.py:5: error: Unsupported operand types for + ("str | None" and "str")
#
# Without --check-untyped-defs, that error is silent — the function is
# unannotated, so mypy never looks inside. With it on, the body is
# checked against inferred types and the None case surfaces.
#
# This is the flag that makes gradual adoption productive: you get
# real errors inside legacy functions before committing to annotating
# every signature.

## Step 4 — scoping `# type: ignore` to specific codes

During migration there will be lines that aren't ready to fix. A bare `# type: ignore` silences *every* error on that line — including ones you'd actually want to know about. Scoping to the specific error code keeps the suppression honest.

In [ ]:
# scoreboard.py — version 3, with scoped ignores
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from typing import reveal_type


def parse_line(line):  # type: ignore[no-untyped-def]
    """Not ready to annotate this yet — but only suppress the no-untyped-def error."""
    name, _, raw = line.partition(":")
    return name.strip(), int(raw)


def tally(lines):
    scores = {}  # type: ignore[var-annotated]
    for line in lines:
        name, points = parse_line(line)
        scores.setdefault(name, []).append(points)
    return scores


def best_player(scores):
    best = None  # type: ignore[assignment]
    best_total = -1
    for name, points in scores.items():
        total = sum(points)
        if total > best_total:
            best_total = total
            best = name
    return best

In [ ]:
# Run:  mypy --check-untyped-defs scoreboard.py
#
# With the scoped ignores, only the targeted errors are suppressed.
# If a NEW error appears on one of these lines — say a wrong-argument
# type — mypy still reports it, because the ignore is pinned to a
# single code.
#
# The placement rule: the ignore comment must be at the start of the
# line's comment section to actually suppress the error. A bare
# `# type: ignore` (no code) silences everything on the line — avoid
# it during migration.
#
# Pair with `warn_unused_ignores = true` so ignores that no longer
# suppress anything get flagged as stale.

## Step 5 — ratchet up with per-module overrides

Once a module is fully annotated, flip `disallow_untyped_defs` on for just that subtree via `[[tool.mypy.overrides]]`. This is the gradual path the mypy docs describe: tighten package by package instead of flipping `--strict` globally on day one. [Source: mypy.readthedocs.io/en/stable/existing_code.html](https://mypy.readthedocs.io/en/stable/existing_code.html)

In [ ]:
# pyproject.toml — gradual strictness via per-module overrides
config = """
[tool.mypy]
python_version = "3.11"
strict = false  # not yet — ratchet per-module instead
check_untyped_defs = true
warn_unused_ignores = true

[[tool.mypy.overrides]]
module = "scoreboard.*"
disallow_untyped_defs = true  # this package is fully annotated now
"""

In [ ]:
# Run:  mypy scoreboard.py
#
# With the override active, any new untyped function added to scoreboard
# fails the check — the package is now held to a higher bar than the rest
# of the project. Other modules still run under the global baseline
# (check_untyped_defs only), so the migration can proceed module by module.
#
# Once every module has its own override, flip `strict = true` globally
# and remove the overrides — the ratchet is complete.

## Verify

The adoption path in sequence:

1. Run `mypy` on the untyped module → confirms the silent-pass gotcha.
2. Add `reveal_type()` probes → confirms inference matches the mental model.
3. Enable `--check-untyped-defs` → surfaces body errors without requiring signatures.
4. Scope `# type: ignore[<code>]` → suppressions stay narrow, paired with `warn_unused_ignores`.
5. Add per-module `[[tool.mypy.overrides]]` → ratchet `disallow_untyped_defs` package by package.

At each step the module gets stricter without a flag-day rewrite. The `reveal_type` calls are temporary — once real annotations replace them, the probes come out and the types are pinned explicitly.